# TCC2 — Pipeline Completo no Google Colab (Sessão Única)

Este notebook reúne todas as etapas em **uma única sessão/kernel** para evitar trocar de notebook e perder a GPU alocada:
1. **Setup**: Google Drive (`TCC_RESULTS_DIR`), Git (`clone`/`pull`), dependências e `sys.path`.
2. **Diagnóstico**: GPU, VRAM, precisão mista (`fp16` na T4 / `bf16` na L4) e `pytest`.
3. **EDA (opcional)**: gera splits fixos, tabelas `.csv`/`.tex` e histogramas.
4. **Validação (opcional)**: `--smoke-test` e `dry_run.yaml`.
5. **Treino (com retomada e ETA)**: grade `main.yaml` (pula runs já com `metrics.json`).
6. **Análise Estatística e Relatório**: gera tabelas, gráficos e `results/summary.md`.
7. **Exportação `.zip`**: compacta `results/` para download.


## 1. Setup do Ambiente, Google Drive e Repositório

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

# Evita conflito do TensorFlow pré-instalado no Colab e warnings de fork do tokenizers
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TORCH"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

IN_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()
USE_DRIVE = True  # Salva todos os resultados no Drive para sobreviver a quedas da sessão
REPO_URL = "https://github.com/juniors719/codeforces-transformers.git"

DRIVE_BASE = Path("/content/drive/MyDrive/tcc2")
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    (DRIVE_BASE / "results").mkdir(parents=True, exist_ok=True)
    os.environ["TCC_RESULTS_DIR"] = str(DRIVE_BASE / "results")
    print("Resultados serão salvos em:", os.environ["TCC_RESULTS_DIR"])
else:
    print("Execução local ou sem Drive. Usando results/ do repositório.")

REPO_DIR = Path("/content/codeforces-transformers") if IN_COLAB else Path.cwd().parent
if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
src_path = str((REPO_DIR / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()

print("Diretório atual:", Path.cwd())
!git log -n 1 --oneline

In [ ]:
# Instala dependências compatíveis com o Colab (mantendo PyTorch CUDA e protobuf nativos)
!pip install -q transformers==4.49.0 datasets==3.3.2 accelerate==1.4.0 scikit-learn==1.6.1 scipy==1.15.2 pandas==2.2.3 matplotlib==3.10.0 PyYAML==6.0.2 tqdm==4.67.1 "sentencepiece>=0.2.0,<0.3.0" pytest==8.3.4
!pip install -q --no-deps --no-build-isolation -e .

## 2. Diagnóstico de GPU e Testes Unitários

In [ ]:
import json
from tcc.utils import get_environment_metadata

!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
print(json.dumps(get_environment_metadata(), indent=2, ensure_ascii=False))
!pytest -q

## 3. Análise Exploratória (EDA) — Opcional
Pule esta seção se já gerou `results/splits/` e as tabelas da EDA.

In [ ]:
import pandas as pd
from tcc.config import load_config
from tcc.data import load_or_prepare_splits
from tcc.eda import (
    analyze_token_lengths_by_tokenizer,
    compute_difficulty_distribution,
    compute_tag_cooccurrence,
    compute_tags_per_problem_stats,
    compute_trivial_baselines,
    plot_difficulty_distribution,
    plot_token_length_histograms,
    propose_input_format_budget,
    save_table_csv_and_tex,
)

RUN_EDA = False  # Altere para True para executar a EDA completa nesta sessão

if RUN_EDA:
    cfg = load_config()
    bundle = load_or_prepare_splits(cfg, force_rebuild=False)
    train_df, val_df, test_df = bundle["train"], bundle["val"], bundle["test"]
    tag_vocab, train_tag_freq = bundle["tag_vocab"], bundle["train_tag_freq"]

    display(pd.DataFrame(bundle["filter_steps"]))
    print("Aliases:", json.dumps(bundle["alias_report"], indent=2, ensure_ascii=False))
    print("Anos:", json.dumps(bundle["year_summary"], indent=2, ensure_ascii=False))

    save_table_csv_and_tex(train_tag_freq, cfg["paths"]["tables_dir"], "eda_tag_frequencies")
    dist_df = compute_difficulty_distribution(train_df, val_df, test_df, cfg["labels"]["difficulty_names"])
    display(dist_df)
    save_table_csv_and_tex(dist_df, cfg["paths"]["tables_dir"], "eda_difficulty_distribution")
    plot_difficulty_distribution(dist_df, cfg["paths"]["figures_dir"])

    all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    overall_len_df, by_class_len_df, raw_lengths = analyze_token_lengths_by_tokenizer(
        all_df, cfg["models"], cfg["tokenization"]["fields"], cfg["labels"]["difficulty_names"], cfg["paths"]["cache_dir"]
    )
    display(overall_len_df)
    print(propose_input_format_budget(overall_len_df, cfg["tokenization"]["reserve_input_format_tokens"]))
    save_table_csv_and_tex(overall_len_df, cfg["paths"]["tables_dir"], "eda_token_lengths_overall")
    save_table_csv_and_tex(by_class_len_df, cfg["paths"]["tables_dir"], "eda_token_lengths_by_class")
    plot_token_length_histograms(raw_lengths, cfg["paths"]["figures_dir"])

    baselines = compute_trivial_baselines(train_df, test_df, tag_vocab, cfg["labels"]["exclude_tag_prefix"])
    print(json.dumps(baselines, indent=2, ensure_ascii=False))
    (Path(cfg["paths"]["tables_dir"]) / "eda_trivial_baselines.json").write_text(
        json.dumps(baselines, indent=2, ensure_ascii=False), encoding="utf-8"
    )
else:
    print("RUN_EDA=False (pulado).")

## 4. Validação Rápida (Smoke Test + Dry-Run) — Opcional

In [ ]:
# Descomente as linhas abaixo para validar memória por arquitetura ou o pipeline dry_run:
# !python -m tcc.runner --smoke-test
# !python -m tcc.runner --config configs/experiments/dry_run.yaml

## 5. Execução da Grade Experimental (Retomada Automática + ETA)
Qualquer run que já tenha `metrics.json` no Drive será pulado automaticamente.

In [ ]:
# Dica: para testar a velocidade primeiro, rode 1 modelo e 1 seed:
# !python -m tcc.runner --config configs/experiments/main.yaml --models bert-base-uncased --seeds 42

# Grade principal completa (70 runs: 5 modelos @ 512 + 2 longos @ 2048 × 10 seeds):
!python -m tcc.runner --config configs/experiments/main.yaml

## 6. Análise Estatística, Tabelas, Figuras e `summary.md`

In [ ]:
from tcc.config import load_config
from tcc.reporting import generate_full_report

cfg = load_config()
report = generate_full_report(cfg, experiment_name="main")
print("Status:", report["status"], "| Runs carregados:", report["n_runs"])
print("Relatório salvo em:", report["summary_md"])

if report["status"] == "ok":
    display(report["summary_df"])
    if not report["kruskal_df"].empty:
        display(report["kruskal_df"])
    if not report["pairwise_df"].empty:
        display(report["pairwise_df"][report["pairwise_df"]["contraste_planejado"] != ""])

## 7. Exportar `results/` como `.zip`

In [ ]:
from tcc.config import load_config
from tcc.utils import zip_results_dir

cfg = load_config()
out_zip = Path("/content/results_tcc.zip") if Path("/content").exists() else Path("results_tcc.zip")
zip_path = zip_results_dir(cfg["paths"]["results_dir"], out_zip)
print("Arquivo gerado:", zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))